# Study Area & Ecological Mask

Defines the study extent and conservative ecological mask for Sudan. The mask excludes areas where *P. orientalis* transmission is essentially impossible (mean annual rainfall < 150mm), following a "where VL certainly isn't" framing rather than trying to define the plausible zone positively. MaxEnt learns everything else from the data.

In [9]:
import ee
import geemap

ee.Initialize(project="sudan-enm")

# Study extent: generous bounding box
study_extent = ee.Geometry.Rectangle([21.5, 8.5, 39, 22.5])  # [west, south, east, north]

# Sudan boundary for visualization
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
sudan = countries.filter(ee.Filter.eq("ADM0_NAME", "Sudan"))

In [2]:
# Visualize study extent against Sudan boundary
check_map = geemap.Map()
check_map.centerObject(study_extent, zoom=5)

# Sudan border
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
sudan = countries.filter(ee.Filter.eq("ADM0_NAME", "Sudan"))

check_map.addLayer(study_extent, {'color': 'blue'}, 'Study extent')
check_map.addLayer(sudan, {'color': 'red'}, 'Sudan border')
check_map

Map(center=[15.588947482443258, 30.250000000000014], controls=(WidgetControl(options=['position', 'transparent…

## Ecological Mask

Conservative rainfall-only threshold at 150mm mean annual precipitation. This excludes the northern desert while retaining Darfur, where recent VL case reports exist (Mohammed et al. 2018). Temperature thresholds from the original version (34–38°C) were too restrictive and excluded known endemic areas.

Rainfall source: CHIRPS Daily v2.0, summed to annual totals for 2000–2024, then averaged.

In [10]:
# CHIRPS daily rainfall, summed to annual totals, averaged across 2000-2024
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(study_extent)

years = ee.List.sequence(2000, 2024)

def annual_rainfall(year):
    year = ee.Number(year)
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year.add(1), 1, 1)
    return chirps.filterDate(start, end).sum().set('year', year)

annual_precip = ee.ImageCollection(years.map(annual_rainfall))
mean_annual_rainfall = annual_precip.mean().clip(study_extent)

# Conservative mask: exclude areas below 150mm mean annual rainfall
ecological_mask = mean_annual_rainfall.gte(150)

# Visualize
mask_map = geemap.Map()
mask_map.centerObject(study_extent, zoom=5)
mask_map.addLayer(mean_annual_rainfall.clip(study_extent),
    {'min': 0, 'max': 1200, 'palette': ['white', 'blue', 'darkblue']},
    'Mean annual rainfall (mm)', opacity=0.6)
mask_map.addLayer(ecological_mask.selfMask().clip(study_extent),
    {'palette': ['green']},
    'Ecological mask (>=150mm)', opacity=0.5)
mask_map.addLayer(sudan, {'color': 'red'}, 'Sudan border', opacity=0.5)
mask_map

Map(center=[15.588947482443258, 30.250000000000014], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
import pandas as pd

occ = pd.read_csv("../data/raw/compiled_vl_presences.csv")  # adjust path
print(f"Total records: {len(occ)}")

# Build points server-side using ee.List
coords = occ[['longitude', 'latitude']].values.tolist()
points = ee.FeatureCollection(
    ee.List(coords).map(lambda c: ee.Feature(ee.Geometry.Point(c)))
)

# Sample the mask at each point
sampled = ecological_mask.sampleRegions(
    collection=points,
    scale=1000
).aggregate_histogram('precipitation').getInfo()

print(f"Mask values at occurrence points: {sampled}")

Total records: 126
Mask values at occurrence points: {'0': 6, '1': 120}


In [12]:
# Add mask value to each point and pull back the ones outside
sampled_fc = ecological_mask.sampleRegions(
    collection=points,
    scale=1000
)

# Get all features and match back to original data
results = sampled_fc.getInfo()
mask_values = [f['properties']['precipitation'] for f in results['features']]

occ['mask_value'] = mask_values
outside = occ[occ['mask_value'] == 0]
print(f"Points outside mask ({len(outside)}):")
print(outside[['latitude', 'longitude', 'source', 'year', 'location_name']].to_string())

Points outside mask (6):
      latitude  longitude                   source  year                        location_name
3    15.548000  32.532000  Pigott_et_al_2014_Dryad  2001                                  NaN
4    15.500000  32.610000  Pigott_et_al_2014_Dryad  2001                                  NaN
7    15.645000  32.476000  Pigott_et_al_2014_Dryad  2001                                  NaN
14   15.580000  32.530000  Pigott_et_al_2014_Dryad  2003                                  NaN
77   15.642337  32.493891        Elnoor_et_al_2024  2020  Tropical Disease Teaching Hospital 
120  15.894590  32.540000   hassan_et_al_2020_fig4  2013                                  NaN


### Mask Validation

6 of 126 occurrence points fall outside the ecological mask. All are clustered around Khartoum (32.5°E, 15.5–15.9°N), which receives ~120–160mm annual rainfall — below the 150mm threshold. These include the Tropical Disease Teaching Hospital, a major VL referral center. The remaining five (from Pigott et al. 2014) have no location names but share the same geographic cluster.

These are treatment-location records, not necessarily transmission-location records — patients infected in the endemic belt (Gedaref, Blue Nile) travel to Khartoum for care. Including them would train the model on conditions where cases are *reported* rather than where transmission *occurs*. They are excluded from model fitting but retained in the compiled dataset with a flag. The 150mm threshold is unchanged.

One of these six points is in North Darfur and falls just outside boundary. 

In [13]:
# Export ecological mask
export_mask = ee.batch.Export.image.toDrive(
    image=ecological_mask,
    description='ecological_mask_150mm',
    folder='sudan_enm_covariates',
    region=study_extent,
    scale=1000,
    crs='EPSG:4326',
    maxPixels=1e9
)
export_mask.start()
print("Mask export started. Check GEE Task Manager.")

Mask export started. Check GEE Task Manager.
